# Data Merge & Processing Strategy

## Objective
Merge historical Parquet data with new update data (JSON/CSV) for Speed Cameras and Traffic Violations. 
This notebook handles schema differences (Column Names/Types) and deduplicates records.

## 🛠 Setup & Data Registration

In [29]:
import duckdb
import os
import pandas as pd

# 1. Connect to In-Memory DB (or file if preferred)
con = duckdb.connect(":memory:")
con.execute("INSTALL httpfs; LOAD httpfs;")

# 2. Define File Paths
DATA_DIR = "../data/opendata"
files = {
    "speed_historic": "nyc_speed_cameras_historic.parquet",
    "speed_test1": "test1_nyc_speed_cameras.json",
    "speed_test2": "test2_nyc_speed_cameras.csv",
    "speed_test3": "test3_nyc_speed_cameras.csv",
    "traffic_historic": "nyc_traffic_violations_historic.parquet",
    "traffic_test1": "test1_nyc_traffic_violations.json",
    "traffic_test2": "test2_nyc_traffic_violations.csv",
    "traffic_test3": "test3_nyc_traffic_violations.csv"
}

# 3. Register Views (Lazy Loading)
print("🚀 Registering Views...")
for name, filename in files.items():
    path = os.path.join(DATA_DIR, filename)
    if os.path.exists(path):
        if ".parquet" in filename: 
            con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{path}')")
        elif ".json" in filename: 
            con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_json_auto('{path}')")
        elif ".csv" in filename: 
            # Force all_varchar=True initially to avoid type errors during sniff
            con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{path}', all_varchar=True)")
        print(f"✅ Registered: {name}")
    else:
        print(f"❌ Missing: {path}")

🚀 Registering Views...
✅ Registered: speed_historic
✅ Registered: speed_test1
✅ Registered: speed_test2
✅ Registered: speed_test3
✅ Registered: traffic_historic
✅ Registered: traffic_test1
✅ Registered: traffic_test2
✅ Registered: traffic_test3


## 🧐 Data Inspection
Let's look at the raw data (`HEAD`) for each file to visually understand the differences in columns and formats (e.g. Title Case vs snake_case, Date formats).

In [30]:
print("\n--- 📸 SPEED CAMERAS (Raw Data Samples) ---")
print("1. Historic (Reference Schema):")
display(con.sql("SELECT * FROM speed_historic LIMIT 3").df())

print("\n2. Test 1 (JSON - Standard-ish):")
display(con.sql("SELECT * FROM speed_test1 LIMIT 3").df())

print("\n3. Test 2 (CSV - Title Case, Split Dates):")
display(con.sql("SELECT * FROM speed_test2 LIMIT 3").df())

print("\n4. Test 3 (CSV - Abbreviated Names):")
display(con.sql("SELECT * FROM speed_test3 LIMIT 3").df())

print("\n--- 🚦 TRAFFIC VIOLATIONS (Raw Data Samples) ---")
print("1. Historic (Reference Schema):")
display(con.sql("SELECT * FROM traffic_historic LIMIT 3").df())

print("\n2. Test 1 (JSON):")
display(con.sql("SELECT * FROM traffic_test1 LIMIT 3").df())

print("\n3. Test 2 (CSV - Title Case):")
display(con.sql("SELECT * FROM traffic_test2 LIMIT 3").df())

print("\n4. Test 3 (CSV - Abbreviated):")
display(con.sql("SELECT * FROM traffic_test3 LIMIT 3").df())


--- 📸 SPEED CAMERAS (Raw Data Samples) ---
1. Historic (Reference Schema):


,issue_date,created_at,amount_due,county,fine_amount,interest_amount,issuing_agency,judgment_entry_date,license_type,payment_amount,penalty_amount,plate,precinct,reduction_amount,state,summons_number,violation,violation_status,violation_time
0,2025-04-13 08:00:00-04:00,2025-04-20 01:53:02.192000-04:00,0.0,BK,50.0,0.00,DEPARTMENT OF TRANSPORTATION,NaT,PAS,50.00,0.0,HVV8423,0,0.0,NY,4943625630,PHTO SCHOOL ZN SPEED VIOLATION,None,12:06A
1,2023-10-10 08:00:00-04:00,2023-10-16 16:18:20.947000-04:00,0.0,BK,50.0,0.00,DEPARTMENT OF TRANSPORTATION,NaT,PAS,50.00,0.0,D92RJR,0,0.0,NJ,4867038659,PHTO SCHOOL ZN SPEED VIOLATION,None,07:58A
2,2024-11-11 10:00:00-05:00,2024-11-24 02:42:12.312000-05:00,0.0,BX,50.0,0.65,DEPARTMENT OF TRANSPORTATION,2025-02-06 05:00:00-05:00,COM,75.65,25.0,65687NB,0,0.0,NY,4925626346,PHTO SCHOOL ZN SPEED VIOLATION,None,10:37A



2. Test 1 (JSON - Standard-ish):


,issue_date,created_at,amount_due,county,fine_amount,interest_amount,issuing_agency,judgment_entry_date,license_type,payment_amount,penalty_amount,plate,precinct,reduction_amount,state,summons_number,violation,violation_status,violation_time
0,2025-11-02T09:00:00-05:00,2025-11-09T02:39:46.157-05:00,0.0,QN,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,OMS,50.0,0.0,LWS4720,0,0.0,NY,4970472282,PHTO SCHOOL ZN SPEED VIOLATION,None,02:43P
1,2025-11-03T10:00:00-05:00,2025-11-09T02:39:46.157-05:00,0.0,ST,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,PAS,50.0,0.0,K74KDD,0,0.0,NJ,4970523400,PHTO SCHOOL ZN SPEED VIOLATION,None,07:28P
2,2025-11-07T10:00:00-05:00,2025-11-16T03:16:49.685-05:00,0.0,QN,50.0,0.0,DEPARTMENT OF TRANSPORTATION,None,PAS,50.0,0.0,HSE8111,0,0.0,NY,4971127483,PHTO SCHOOL ZN SPEED VIOLATION,None,10:52A



3. Test 2 (CSV - Title Case, Split Dates):


...

,license_id,county,age,birth_date,violation_code,violation_year,violation_month,points
0,BR425170,Westchester,70,1954-12-07,SPEED IN ZONE 21-30,2025,11,6
1,PK119730,Other,30,1994-12-07,CHILD SAFETY RESTRAINT,2025,11,3
2,JW166375,Nassau,19,2005-12-06,SPEED IN ZONE 31-40,2025,11,8



3. Test 2 (CSV - Title Case):


,License Id,County,Age,Violation Code,Violation Year,Violation Month,Points,Birth Year,Birth Month
0,BR425170,Westchester,70,SPEED IN ZONE 21-30,2025,11,6,1954,12
1,PK119730,Other,30,CHILD SAFETY RESTRAINT,2025,11,3,1994,12
2,JW166375,Nassau,19,SPEED IN ZONE 31-40,2025,11,8,2005,12



4. Test 3 (CSV - Abbreviated):


,lic_id,county,age,v_code,v_year,v_month,points,dob_formatted,meta_sys_version
0,BR425170,Westchester,70,SPEED IN ZONE 21-30,2025,11,6,07/12/1954,v3_legacy_export
1,PK119730,Other,30,CHILD SAFETY RESTRAINT,2025,11,3,07/12/1994,v3_legacy_export
2,JW166375,Nassau,19,SPEED IN ZONE 31-40,2025,11,8,06/12/2005,v3_legacy_export


## 🔍 Feature Analysis (Schema Inspection)
Comparing columns and types across datasets to identify what needs normalization.

In [31]:
def show_features(tables):
    schema_data = []
    for t in tables:
        try:
            df = con.sql(f"DESCRIBE {t}").df()
            for _, row in df.iterrows():
                schema_data.append({"Table": t, "Column": row['column_name'], "Type": row['column_type']})
        except: pass
    
    if not schema_data: return
    
    # Pivot for easy comparison
    all_features = pd.DataFrame(schema_data)
    pivoted = all_features.pivot_table(index="Column", columns="Table", values="Type", aggfunc='first').fillna("MISSING")
    display(pivoted)

print("--- Speed Camera Features ---")
show_features([t for t in files.keys() if "speed" in t])

print("\n--- Traffic Violation Features ---")
show_features([t for t in files.keys() if "traffic" in t])

--- Speed Camera Features ---


Table,speed_historic,speed_test1,speed_test2,speed_test3
Column,,,,
Amount Due,MISSING,MISSING,VARCHAR,MISSING
County,MISSING,MISSING,VARCHAR,MISSING
Created At,MISSING,MISSING,VARCHAR,MISSING
Fine Amount,MISSING,MISSING,VARCHAR,MISSING
Interest Amount,MISSING,MISSING,VARCHAR,MISSING
Issue Day,MISSING,MISSING,VARCHAR,MISSING
Issue Month,MISSING,MISSING,VARCHAR,MISSING
Issue Year,MISSING,MISSING,VARCHAR,MISSING
Issuing Agency,MISSING,MISSING,VARCHAR,MISSING



--- Traffic Violation Features ---


Table,traffic_historic,traffic_test1,traffic_test2,traffic_test3
Column,,,,
Age,MISSING,MISSING,VARCHAR,MISSING
Birth Month,MISSING,MISSING,VARCHAR,MISSING
Birth Year,MISSING,MISSING,VARCHAR,MISSING
County,MISSING,MISSING,VARCHAR,MISSING
License Id,MISSING,MISSING,VARCHAR,MISSING
Points,MISSING,MISSING,VARCHAR,MISSING
Violation Code,MISSING,MISSING,VARCHAR,MISSING
Violation Month,MISSING,MISSING,VARCHAR,MISSING
Violation Year,MISSING,MISSING,VARCHAR,MISSING


## ⚙️ Strategy 1: Speed Cameras Merge
**Goal Schema**: Aligned with `speed_historic`.

**Transformations**:
- **Dates**: Converting text/parts to `TIMESTAMP WITH TIME ZONE` (or DATE).
- **Columns**: Standardizing names (snake_case).
- **Types**: Explicitly casting amounts to `DOUBLE` and IDs to `BIGINT`.

In [32]:
# 1. Normalize TEST 1 (JSON)
# Needs: created_at -> TIMESTAMPTZ, judgment -> TIMESTAMPTZ
sql_speed_1 = """
SELECT 
    *, 
    try_cast(created_at AS TIMESTAMP WITH TIME ZONE) as created_at_cast,
    try_cast(judgment_entry_date AS TIMESTAMP WITH TIME ZONE) as judgment_entry_date_cast,
    try_cast(issue_date AS TIMESTAMP WITH TIME ZONE) as issue_date_cast
FROM speed_test1
"""
# We create a view that projects ONLY the columns we want, aliased correctly if needed.
# Note: JSON usually infers correctly, but we ensure types match historic.
con.execute(f"""CREATE OR REPLACE VIEW norm_speed_test1 AS 
SELECT 
    summons_number,
    plate,
    state,
    license_type,
    issue_date_cast as issue_date,
    violation_time,
    violation,
    judgment_entry_date_cast as judgment_entry_date,
    fine_amount,
    penalty_amount,
    interest_amount,
    reduction_amount,
    payment_amount,
    amount_due,
    precinct,
    county,
    issuing_agency,
    violation_status,
    created_at_cast as created_at
FROM ({sql_speed_1})
""")

# 2. Normalize TEST 2 (CSV - Title Case)
# Needs: Title Case -> snake_case, Date Parts -> DATE
con.execute("""CREATE OR REPLACE VIEW norm_speed_test2 AS
SELECT
    "Summons Number" as summons_number,
    "Plate" as plate,
    "State" as state,
    "License Type" as license_type,
    make_timestamp("Issue Year" :: int, "Issue Month" :: int, "Issue Day" :: int, 0, 0, 0) as issue_date, 
    "Violation Time" as violation_time,
    "Violation" as violation,
    try_cast("Judgment Entry Date" AS TIMESTAMP WITH TIME ZONE) as judgment_entry_date,
    "Fine Amount" :: double as fine_amount,
    "Penalty Amount" :: double as penalty_amount,
    "Interest Amount" :: double as interest_amount,
    "Reduction Amount" :: double as reduction_amount,
    "Payment Amount" :: double as payment_amount,
    "Amount Due" :: double as amount_due,
    NULL :: bigint as precinct,
    "County" as county,
    "Issuing Agency" as issuing_agency,
    "Violation Status" as violation_status,
    try_cast("Created At" AS TIMESTAMP WITH TIME ZONE) as created_at
FROM speed_test2
""")

# 3. Normalize TEST 3 (CSV - Different Names)
# Needs: issued_date -> issue_date
con.execute("""CREATE OR REPLACE VIEW norm_speed_test3 AS
SELECT
    summons_number,
    plate,
    state,
    license_type,
    try_cast(issued_date AS TIMESTAMP WITH TIME ZONE) as issue_date,
    violation_time,
    violation,
    NULL :: timestamp with time zone as judgment_entry_date,
    fine_amount :: double as fine_amount,
    penalty_amount :: double as penalty_amount,
    interest_amount :: double as interest_amount,
    reduction_amount :: double as reduction_amount,
    payment_amount :: double as payment_amount,
    amount_due :: double as amount_due,
    precinct :: bigint as precinct,
    county,
    issuing_agency,
    violation_status,
    try_cast(created_at AS TIMESTAMP WITH TIME ZONE) as created_at
FROM speed_test3
""")

print("✅ Speed Camera Normalization Views Created.")

✅ Speed Camera Normalization Views Created.


In [33]:
# MERGE Speed Cameras using UNION ALL BY NAME
# UNION ALL BY NAME matches columns automatically, filling missing ones with NULL.
# This fixes the 'BIGINT -> TIMESTAMP' error because it won't align wrong columns.

query = """
CREATE OR REPLACE TABLE speed_cameras_final AS 
SELECT DISTINCT ON (summons_number) * 
FROM (
    SELECT * FROM speed_historic
    UNION ALL BY NAME
    SELECT * FROM norm_speed_test1
    UNION ALL BY NAME
    SELECT * FROM norm_speed_test2
    UNION ALL BY NAME
    SELECT * FROM norm_speed_test3
)
ORDER BY summons_number, created_at DESC -- Keep latest record if duplicated
"""
con.execute(query)
print("🎉 Speed Cameras Merged Successfully!")
print("Rows:", con.execute("SELECT COUNT(*) FROM speed_cameras_final").fetchone()[0])

🎉 Speed Cameras Merged Successfully!
Rows: 1972289


## ⚙️ Strategy 2: Traffic Violations Merge

In [34]:
# Normalize Test 2 (Title Case, Birth Parts)
con.execute("""CREATE OR REPLACE VIEW norm_traffic_test2 AS
SELECT 
    NULL as license_id,
    make_date("Birth Year" :: int, "Birth Month" :: int, 1) as birth_date,
    "Age" :: bigint as age,
    "Violation Code" as violation_code,
    "Violation Year" :: bigint as violation_year,
    "Violation Month" :: bigint as violation_month,
    "Points" :: bigint as points,
    "County" as county
FROM traffic_test2
""")

# Normalize Test 3 (Abbreviated)
con.execute("""CREATE OR REPLACE VIEW norm_traffic_test3 AS
SELECT 
    lic_id as license_id,
    try_cast(dob_formatted AS DATE) as birth_date,
    NULL :: bigint as age,
    v_code as violation_code,
    v_year :: bigint as violation_year,
    v_month :: bigint as violation_month,
    points :: bigint as points,
    county
FROM traffic_test3
""")

# MERGE Traffic Violations
query = """
CREATE OR REPLACE TABLE traffic_violations_final AS 
SELECT DISTINCT * 
FROM (
    SELECT * FROM traffic_historic
    UNION ALL BY NAME
    SELECT * FROM traffic_test1
    UNION ALL BY NAME
    SELECT * FROM norm_traffic_test2
    UNION ALL BY NAME
    SELECT * FROM norm_traffic_test3
)
"""
con.execute(query)
print("🎉 Traffic Violations Merged Successfully!")
print("Rows:", con.execute("SELECT COUNT(*) FROM traffic_violations_final").fetchone()[0])

🎉 Traffic Violations Merged Successfully!
Rows: 1053552


## 🚗 Speed Camera Engine: Vehicle Aggregation
Creating the **Vehicle Summary Table** to identify high-risk speeders.

**Logic**:
1.  **Window**: Last 12 months from the latest available date (`as_of_date`).
2.  **Aggregation**: Group by `plate` + `state`.
3.  **Metrics**: Count violations (`violations_12m`), sum fines, track dates.
4.  **Status**:
    - **TRIGGER** 🔴: >= 16 violations
    - **WARNING** 🟡: 12-15 violations
    - **OK** 🟢: < 12 violations

In [35]:
# 1. Determine Analysis Window
# Find the latest date in our merged data to act as "today" for simulation purposes
res = con.execute("SELECT MAX(issue_date) FROM speed_cameras_final").fetchone()
as_of_date = pd.Timestamp(res[0])
cutoff_date = as_of_date - pd.DateOffset(months=12)

print(f"📅 As Of Date (Latest Data): {as_of_date.date()}")
print(f"🔙 Cutoff Date (12m Window): {cutoff_date.date()}")

# 2. Calculate Vehicle Summary
engine_sql = f"""
CREATE OR REPLACE TABLE vehicle_speed_summary AS
WITH filtered AS (
    SELECT * FROM speed_cameras_final
    WHERE issue_date >= '{cutoff_date}'
),
aggregated AS (
    SELECT
        plate,
        state,
        COUNT(DISTINCT summons_number) as violations_12m,
        MIN(issue_date) as first_violation_12m,
        MAX(issue_date) as last_violation_12m,
        arg_max(county, issue_date) as county_last_seen, -- Function to get county from latest violation
        SUM(fine_amount) as total_fines_12m
    FROM filtered
    GROUP BY plate, state
)
SELECT
    row_number() OVER (ORDER BY violations_12m DESC, plate) as vehicle_id, -- Sequential Key
    plate,
    state,
    county_last_seen,
    violations_12m,
    first_violation_12m,
    last_violation_12m,
    total_fines_12m,
    CASE
        WHEN violations_12m >= 16 THEN 'TRIGGER'
        WHEN violations_12m >= 12 THEN 'WARNING'
        ELSE 'OK'
    END as status,
    16 as trigger_threshold,
    12 as warning_lower_bound,
    CAST('{as_of_date}' AS DATE) as as_of_date
FROM aggregated
ORDER BY violations_12m DESC;
"""

con.execute(engine_sql)
print("✅ Vehicle Summary Table Created.")

# Check stats
stats = con.execute("SELECT status, COUNT(*) FROM vehicle_speed_summary GROUP BY status").fetchall()
print("\n📊 Status Distribution:")
for s, c in stats:
    print(f"  {s}: {c}")

📅 As Of Date (Latest Data): 2025-10-14
🔙 Cutoff Date (12m Window): 2024-10-14
✅ Vehicle Summary Table Created.

📊 Status Distribution:
  OK: 474690
  TRIGGER: 76


In [36]:
# 3. JDBC/Export Simulation
# Export to CSV for the dashboard to pick up
export_path = "../data/exports/vehicle_speed_summary.csv"
os.makedirs("../data/exports", exist_ok=True)

con.execute(f"COPY vehicle_speed_summary TO '{export_path}' (HEADER, DELIMITER ',')")
print(f"\n💾 Exported Engine Results to: {export_path}")

# Show top violators
print("\n🚨 TOP VIOLATORS (Last 12 Months):")
display(con.sql("SELECT * FROM vehicle_speed_summary WHERE status='TRIGGER' LIMIT 5").df())


💾 Exported Engine Results to: ../data/exports/vehicle_speed_summary.csv

🚨 TOP VIOLATORS (Last 12 Months):


,vehicle_id,plate,state,county_last_seen,violations_12m,first_violation_12m,last_violation_12m,total_fines_12m,status,trigger_threshold,warning_lower_bound,as_of_date
0,1,LCM8254,NY,BK,48,2024-10-15 08:00:00-04:00,2025-06-11 08:00:00-04:00,2400.0,TRIGGER,16,12,2025-10-14
1,2,LHW5598,NY,BX,36,2024-10-21 08:00:00-04:00,2025-10-04 08:00:00-04:00,1800.0,TRIGGER,16,12,2025-10-14
2,3,MHW9481,PA,BK,36,2024-10-14 08:00:00-04:00,2025-02-15 10:00:00-05:00,1800.0,TRIGGER,16,12,2025-10-14
3,4,TLN8692,VA,BK,36,2024-10-16 08:00:00-04:00,2025-10-10 08:00:00-04:00,1800.0,TRIGGER,16,12,2025-10-14
4,5,KXM7078,NY,BX,33,2024-11-24 10:00:00-05:00,2025-10-04 08:00:00-04:00,1650.0,TRIGGER,16,12,2025-10-14
